# 05 — Emotion Detection

Uses **DeepFace** with pre-trained emotion model (no training needed).  
DeepFace auto-downloads model weights on first run.

Demo: run emotion detection on real faces from `classification_data`.

In [ ]:
import os
import random
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter
from deepface import DeepFace

DATA_ROOT = Path("../data")
EMOTIONS  = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]

EMOTION_ICONS = {
    "angry": "😠", "disgust": "🤢", "fear": "😨",
    "happy": "😊", "sad": "😢", "surprise": "😲", "neutral": "😐",
}

print("DeepFace loaded.")

## Inference Helper

In [ ]:
def predict_emotion(img_rgb: np.ndarray) -> tuple[str, float]:
    """Run DeepFace emotion detection on an RGB image."""
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    try:
        result = DeepFace.analyze(bgr, actions=["emotion"],
                                  enforce_detection=False, silent=True)
        r = result[0] if isinstance(result, list) else result
        dominant = r["dominant_emotion"]
        conf     = r["emotion"][dominant] / 100.0
        return (dominant, conf)
    except Exception:
        return ("neutral", 0.0)

## Demo on Sample Faces

In [ ]:
all_imgs = list((DATA_ROOT / "classification_data" / "val_data").glob("**/*.jpg"))
samples  = random.sample(all_imgs, 10)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, path in zip(axes.flat, samples):
    img = np.array(Image.open(path).convert("RGB"))
    label, conf = predict_emotion(img)
    icon = EMOTION_ICONS.get(label, "")
    ax.imshow(img)
    ax.set_title(f"{icon} {label}\n{conf:.1%}", fontsize=8)
    ax.axis("off")

plt.suptitle("DeepFace Emotion Detection on Sample Faces", fontsize=12)
plt.tight_layout()
plt.show()

## Emotion Distribution on Val Set

In [ ]:
sample200 = random.sample(all_imgs, min(200, len(all_imgs)))
emotion_counts = Counter()

for path in sample200:
    img   = np.array(Image.open(path).convert("RGB"))
    label, _ = predict_emotion(img)
    emotion_counts[label] += 1

labels = list(emotion_counts.keys())
values = list(emotion_counts.values())

plt.figure(figsize=(8, 4))
plt.bar([f"{EMOTION_ICONS.get(l,'')} {l}" for l in labels], values, color="steelblue")
plt.title("Emotion distribution on 200 sample faces")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

for emo, cnt in sorted(emotion_counts.items(), key=lambda x: -x[1]):
    print(f"  {EMOTION_ICONS.get(emo,'')} {emo:10s}: {cnt}")

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        all_preds.extend(model(imgs).argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=EMOTIONS))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Emotion Detection Confusion Matrix")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
